# Reach all 22 scheduled languages from one clip

One recorded advisory, every language in the Eighth Schedule of the Constitution of India.

Pipeline overview:

1. Read the capability roster out of the SDK's own language lists, rather than typing one.
2. Give each of the 22 languages a tier: a full dub, or a translated subtitle track.
3. Dub tier: create the job, upload the media to the signed URL, start it, poll the export status, download.
4. Subtitle tier: transcribe the clip, translate each cue, write an SRT file into `outputs/`.

The dubbing endpoint covers 11 of the 22 languages. The other 11 are reached by speech to text
plus translation, which cover all 22 between them. Nobody is left out.

**This notebook has not been run against the live API.** There was no API key on the machine it
was written on, so every code cell below ships with empty output. Run it yourself before trusting
any of it.

In [ ]:
%pip install -r requirements.txt

## Three things a hardcoded language list gets wrong

1. **The dubbing endpoint does not cover all 22 scheduled languages.** It covers 11. A hardcoded
   list quietly produces nothing for Urdu, Nepali, Santali, Kashmiri, Bodo, Dogri, Konkani,
   Maithili, Manipuri, Sanskrit and Sindhi.
2. **Capability is per endpoint, and it is not a hierarchy.** Assamese can be dubbed and cannot be
   spoken by the standalone text-to-speech endpoint. Any fallback built on "if it can be dubbed it
   can be spoken" fails for Assamese only, in production only.
3. **Odia has two spellings.** `or-IN` at the dubbing endpoint, `od-IN` almost everywhere else. One
   hardcoded `od-IN` sent to `dubbing.create` is a 400 that reads like a broken API.

So the roster and the language codes both come from `video_reach.py`, which reads the SDK's own
`typing.Literal` sets. Nothing about coverage is typed by hand, here or there.

In [ ]:
from __future__ import annotations

import os
import time
import wave
from pathlib import Path

import httpx
from dotenv import load_dotenv
from sarvamai import SarvamAI

from video_reach import (
    STT_MODEL,
    STT_REST_MAX_SECONDS,
    TRANSLATE_MODEL,
    Clip,
    Cue,
    Endpoint,
    Segment,
    Tier,
    build_roster,
    compose_plan,
    coverage_counts,
    pack_segments,
    plan_summary,
    roster_markdown_table,
    to_endpoint_code,
    write_srt,
)

load_dotenv()

if not os.environ.get("SARVAM_API_KEY"):
    raise RuntimeError(
        "Set SARVAM_API_KEY in your environment or in a .env file before running."
    )

# The key is passed explicitly on purpose. The client's api_subscription_key
# default is os.getenv("SARVAM_API_KEY") evaluated once, when sarvamai is
# imported, so a load_dotenv() after the import is already too late.
client = SarvamAI(api_subscription_key=os.environ["SARVAM_API_KEY"])

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
print("client ready")

## The roster, derived

`build_roster()` reads four language lists out of the installed SDK and scores the 22 scheduled
languages against them. If a future release adds a language to the dubbing endpoint, this table
moves on its own.

In [ ]:
roster = build_roster()
counts = coverage_counts(roster)

print(roster_markdown_table(roster))
print()
print(
    f"of the 22 scheduled languages: dubbing {counts.dubbing}, "
    f"translate {counts.translate}, speech to text {counts.speech_to_text}, "
    f"text to speech {counts.text_to_speech}"
)

assamese = next(row for row in roster if row.language.code == "as-IN")
print(
    f"as-IN dubbing={assamese.dubbing} text_to_speech={assamese.text_to_speech}"
    "  <- dubbable, not speakable"
)

## Odia is spelled two ways

Ten language lists in the SDK spell Odia `od-IN`. Exactly two spell it `or-IN`: dubbing, and
realtime streaming speech to text. No list accepts both, so there is no spelling that works
everywhere and the mapper has to be keyed on the endpoint.

"Streaming" is not one bucket either: realtime streaming speech to text wants `or-IN`, while
text-to-speech streaming wants `od-IN`.

The repository's own rules file lists `or-IN` as valid for text to speech and speech to text,
where the API accepts only `od-IN`. That is issue #157, reported and still open. This recipe works
correctly against the rules file as it stands and does not edit it.

In [ ]:
for endpoint in Endpoint:
    print(f"{endpoint.name:<14} {to_endpoint_code('od-IN', endpoint)}")

## The clip

The default is the English sample already tracked in this repository, 12.70 seconds long. No new
media ships with this recipe.

Two constraints on a clip of your own:

- The subtitle tier transcribes over the REST endpoint, which is for clips **under 30 seconds**.
  Longer clips need the batch speech-to-text API, which this recipe does not cover.
- The dub tier needs a source language the dubbing endpoint accepts. If the clip's language is not
  one of them, every language falls back to the subtitle tier and all 22 are still reached.

**Dubbing from an audio file is documented but unverified here.** The SDK's own docstring says
`dubbing.create` takes "a source video or audio file", and `export_options` lists `audio` as a
first-class output. We could not confirm it: there was no API key, and nothing about the uploaded
bytes is checked locally. If you have a video file, point `CLIP_PATH` and `CLIP_MIME` at it and you
are on the documented path. The subtitle tier is unaffected either way, because it uses only
speech to text and translation, and both take audio.

In [ ]:
CLIP_PATH = Path("../../sample_data/stt/audio3_en.wav")
CLIP_MIME = "audio/wav"
CLIP_LANGUAGE = "en-IN"

# Dubbing from audio rather than video is documented but unverified here; see
# the cell above. Point these three constants at your own video/mp4 file to
# stay on the path the SDK documents.
if CLIP_PATH.suffix.lower() == ".wav":
    with wave.open(str(CLIP_PATH)) as handle:
        clip_seconds = handle.getnframes() / handle.getframerate()
else:
    raise RuntimeError(
        "Set clip_seconds by hand for a clip that is not a WAV file: the stdlib "
        "wave module reads WAV only, and this recipe adds no media dependency."
    )

print(f"{CLIP_PATH.name}: {clip_seconds:.2f} s (REST ceiling {STT_REST_MAX_SECONDS} s)")

clip = Clip(
    path=CLIP_PATH,
    source_language=CLIP_LANGUAGE,
    duration_seconds=clip_seconds,
    mime_type=CLIP_MIME,
)

## The plan

`compose_plan` turns the roster and the clip into the whole 22-language plan: the tier for each
language, the endpoint-correct code, the calls to make and the artifacts to expect. It performs
none of them, and it needs no API key.

In [ ]:
plan = compose_plan(clip)
print(plan_summary(plan))
print()
for language_plan in plan.language_plans:
    methods = ", ".join(call.method for call in language_plan.calls)
    print(
        f"{language_plan.language.code:<7} {language_plan.tier.value:<8} "
        f"{'+'.join(language_plan.artifacts):<16} {methods}"
    )

## Dub tier

Four things about the dubbing lifecycle that are easy to get wrong:

1. **`dubbing.create` accepts no media.** It returns a job id and a short-lived signed
   `upload_url`. The bytes go there by `PUT`, with **both** `Content-Type: <mime>` and
   `x-ms-blob-type: BlockBlob`. Leaving the second header out is a silent upload failure that
   surfaces much later as a failed job.
2. **`editor_flow=False`, explicitly.** With `editor_flow=True` the pipeline completes but
   auto-export is suppressed and the export status stays empty, which looks like a broken job.
3. **`get_export_status` is the source of truth for downloads**, not `get_live_status`.
   `get_live_status` is a progress signal, and it reports a single `export` field for a one-target
   job and an `exports` list for a multi-target one. The export status endpoint always returns a
   list, one row per language and export type, each row carrying its own status and download URL.
4. **Every field of every dubbing response is optional** and the models allow unknown extras, so
   the SDK guarantees nothing came back. Read every field defensively.

`voice_cloning` and `voice_id` are real parameters on `create` and this recipe never uses them.
Cloning a voice needs consent from the person whose voice it is, and a cookbook has nowhere to
record that, so demonstrating it would teach the wrong default.

One `create` call carries every dub target, so this is one job and not eleven.

In [ ]:
dub_targets = list(plan.dub_job_targets)
print(f"one dubbing job with {len(dub_targets)} targets: {', '.join(dub_targets)}")

created = client.dubbing.create(
    source_language_code=to_endpoint_code(clip.source_language, Endpoint.DUBBING),
    target_language_codes=dub_targets,
    export_options=["video", "audio", "srt"],
    editor_flow=False,
    job_name="all-languages-video-reach",
)

job_data = getattr(created, "data", None)
job_id = getattr(job_data, "job_id", None)
upload_url = getattr(job_data, "upload_url", None)
if not job_id or not upload_url:
    raise RuntimeError(f"dubbing.create returned no job id or upload URL: {created}")

print(f"job {job_id}")

In [ ]:
media_bytes = CLIP_PATH.read_bytes()

upload = httpx.put(
    upload_url,
    content=media_bytes,
    headers={"Content-Type": CLIP_MIME, "x-ms-blob-type": "BlockBlob"},
    timeout=300.0,
)
upload.raise_for_status()
print(f"uploaded {len(media_bytes)} bytes, status {upload.status_code}")

started = client.dubbing.start(job_id)
print(getattr(getattr(started, "data", None), "status", None))

In [ ]:
POLL_SECONDS = 20
MAX_POLLS = 90
FINISHED = {"completed", "failed"}

# The export rows do not name the file container, so these extensions are a
# convenience. Check the saved file if a player refuses it.
SUFFIXES = {"video": "mp4", "audio": "wav", "srt": "srt"}

ready: dict[tuple[str, str], str] = {}
for attempt in range(MAX_POLLS):
    export_status = client.dubbing.get_export_status(job_id)
    rows = getattr(getattr(export_status, "data", None), "exports", None) or []
    for row in rows:
        download_url = getattr(row, "download_url", None)
        if getattr(row, "status", None) == "completed" and download_url:
            key = (
                getattr(row, "target_language", None),
                getattr(row, "export_type", None),
            )
            ready[key] = download_url
    pending = [row for row in rows if getattr(row, "status", None) not in FINISHED]
    print(
        f"poll {attempt + 1}: {len(ready)} ready, {len(pending)} pending, "
        f"{len(rows)} export row(s) in total"
    )
    if rows and not pending:
        break
    time.sleep(POLL_SECONDS)

for row in rows:
    if getattr(row, "status", None) == "failed":
        print(
            f"failed: {getattr(row, 'target_language', None)} "
            f"{getattr(row, 'export_type', None)}"
        )

# Signed download URLs expire in about a day. Read the export status again for
# a fresh one rather than keeping these.
for (language, export_type), url in sorted(ready.items()):
    destination = OUTPUT_DIR / f"dub_{language}.{SUFFIXES.get(export_type, 'bin')}"
    destination.write_bytes(httpx.get(url, timeout=300.0).content)
    print(f"saved {destination}")

## Subtitle tier

This is the half of the product that reaches the other eleven languages, and it is where the
recipe spends most of its care.

- The speech-to-text model is `saaras:v3`. A newer model exists but is not in this repository's
  allowlist, so using it would fail the repository's own strict validation.
- Timestamps are **chunk level**, not word level. The response field is named `words`, and each
  entry is a phrase or a sentence. `pack_segments` therefore splits over-long phrases with
  interpolated times and merges over-short ones; it never assembles words.
- The translate model is `sarvam-translate:v1`, the only one that reaches all 22 scheduled
  languages. `mayura:v1` reaches 12, and its automatic `auto` source detection is a mayura-only
  feature, so the source language is always passed explicitly.
- The cue budget, 42 characters over at most 2 lines, is **our choice**: it is the conventional
  broadcast reading budget, not a limit the API imposes. Change the constants in `video_reach.py`
  if your players want something else.
- The subtitle file is written with real line breaks. A writer that emits the two characters
  backslash and n produces a one-line file that no player will read.

This cell makes one translate call per cue per language, so eleven languages times the cue count.
Start with one language while you are trying it out.

In [ ]:
with CLIP_PATH.open("rb") as handle:
    transcription = client.speech_to_text.transcribe(
        file=handle,
        model=STT_MODEL,
        language_code=to_endpoint_code(clip.source_language, Endpoint.STT),
        with_timestamps=True,
    )

timestamps = getattr(transcription, "timestamps", None)
if timestamps is None:
    raise RuntimeError(
        "the transcription carried no timestamps; ask for with_timestamps=True "
        "and check the clip is under the REST ceiling"
    )

# Every field on every response model is optional, so read them defensively.
phrases = getattr(timestamps, "words", None) or []
starts = getattr(timestamps, "start_time_seconds", None) or []
ends = getattr(timestamps, "end_time_seconds", None) or []

segments = [
    Segment(text=text, start=start, end=end)
    for text, start, end in zip(phrases, starts, ends)
]
cues = pack_segments(segments)
print(f"{len(segments)} chunk-level phrase(s) packed into {len(cues)} cue(s)")
print(getattr(transcription, "transcript", ""))

In [ ]:
# TRANSLATE_MODEL is "sarvam-translate:v1": formal mode only, 2000 input
# characters, and the only model that reaches all 22 scheduled languages.
source_for_translate = to_endpoint_code(clip.source_language, Endpoint.TRANSLATE)
subtitle_plans = [
    language_plan
    for language_plan in plan.language_plans
    if language_plan.tier is Tier.SUBTITLE
]

for language_plan in subtitle_plans:
    target = to_endpoint_code(language_plan.language.code, Endpoint.TRANSLATE)
    translated: list[Cue] = []
    for cue in cues:
        response = client.text.translate(
            input=cue.text,
            source_language_code=source_for_translate,
            target_language_code=target,
            model=TRANSLATE_MODEL,
            mode="formal",
        )
        translated.append(
            Cue(
                index=cue.index,
                start=cue.start,
                end=cue.end,
                text=getattr(response, "translated_text", "") or "",
            )
        )
    destination = OUTPUT_DIR / f"subtitles_{language_plan.language.code}.srt"
    write_srt(translated, destination)
    print(f"{language_plan.language.code}: {len(translated)} cue(s) -> {destination}")

## What this recipe deliberately does not do

- **Voice cloning.** The parameters exist on `create`; consent does not exist in a cookbook.
- **Text to speech anywhere in the pipeline.** It covers 10 of the 22 scheduled languages, and
  reaching for it here would teach exactly the hierarchy assumption that fails for Assamese.
- **The batch speech-to-text API and speaker diarization.** Both are needed for clips over 30
  seconds and for per-speaker subtitles. Separate recipe.
- **Muxing or burning the subtitles into the picture.** That needs a media toolchain this recipe
  does not add.

If you run this against the live API and something here is wrong, the fix belongs in a pull
request against this notebook. Nothing in it has been executed.